# Hamiltonian simulation

Trotterize a transverse-field Ising chain and compare an observable trajectory.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
times = np.linspace(0.0, 1.2, 13)
observable = SparsePauliOp("IIZ")

def evolution_circuit(time_value):
    circuit = QuantumCircuit(3)
    circuit.x(0)
    steps = 6
    dt = float(time_value) / steps
    for _ in range(steps):
        circuit.rzz(1.1 * dt, 0, 1)
        circuit.rzz(1.1 * dt, 1, 2)
        for wire in range(3):
            circuit.rx(0.7 * dt, wire)
    return circuit

circuits = [evolution_circuit(value) for value in times]

def reference_trajectory():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, observable)]).result()[0].data.evs.item() for c in circuits])

reference, reference_ms, _ = benchmark(reference_trajectory)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_trajectory():
    return np.asarray([estimator.run([(c, observable)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_trajectory)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(estimator)
tutorial_result = emit_result(
    notebook="qiskit/12_hamiltonian_simulation.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="observable trajectory atol=3e-6",
    passed=error <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_observable_error": error, "times": times, "reference": reference, "mettleq": candidate},
)